___
<h1> Machine Learning </h1>
<h2> M. Sc. in Electrical and Computer Engineering </h2>
<h3> Instituto Superior de Engenharia / Universidade do Algarve </h3>

[MEEC](https://ise.ualg.pt/en/curso/1477) / [ISE](https://ise.ualg.pt) / [UAlg](https://www.ualg.pt)

Pedro J. S. Cardoso (pcardoso@ualg.pt)
___

# Grid Search
## Simple Grid Search (Manual)

A model such as an SVM has hyperparameters (here `C` and `gamma`) that are not learned from the data. Grid search tries every pair from a list of candidate values and keeps the pair with the best score.

Before using `GridSearchCV`, do the same thing by hand: two nested loops, one over `gamma` and one over `C`. That makes the search easy to see, and it also makes the next mistake easy to see: scoring on the test set while you still choose the hyperparameters.cycles so the hyperparameters vary in some set of values

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

import pandas as pd
import matplotlib.pyplot as plt

Let us start by doing it without a validation set

In [ ]:
digits = load_digits()

X_train, X_test, y_train, y_test = train_test_split(
    digits.data,
    digits.target, 
    random_state=42
)

Find the best parameters according with the training and validation sets

In [ ]:
best_score = 0

list_of_params = [.001, 0.01, 0.1, 1, 10, 100]

for gamma in list_of_params:
    for C in list_of_params:
        print(f"Testing gamma: {gamma}, C: {C} ... ", end="")
        
        svm = SVC(gamma=gamma, C=C).fit(X_train, y_train)
        
        score = svm.score(X_test, y_test)
        print(f"Score: {score}")
        
        if score > best_score:
            print(f"New best score: {score}")
            best_model = svm
            best_score = score
            best_params = {'C': C, 'gamma': gamma}

print(f'Over the test set, the best params are: {best_params} with a score of: {best_score}')

## The Danger of Overfitting the Parameters and the Validation Set

We tried many different parameters and selected the one with best accuracy on the test set, but this accuracy won’t necessarily carry over to new data. Because we used the test data to adjust the parameters, we can no longer use it to assess how good the model is. 

This is the same reason we needed to split the data into training and test sets in the first place; we need an independent
dataset to evaluate, one that was not used to create the model.

Let us start by building a training (60%), a testing (20%), and a validation (20%) set


| TRAIN           | VALIDATE                                 | TEST                        |
|-----------------|------------------------------------------|-----------------------------|
| 60%              | 20%                                      | 20%                         |
| X_train          | X_validate                               | X_test                      |
| y_train          | y_validate                               | y_test                      |


In [ ]:
digits = load_digits()

X_train_validate, X_test, y_train_validate, y_test = train_test_split(
    digits.data,
    digits.target, 
    test_size=.2,
    random_state=42
)

X_train, X_validate, y_train, y_validate = train_test_split(
    X_train_validate,
    y_train_validate, 
    test_size=.25, # this will give 20% of the original set
    random_state=42
)

print(f"Shapes of the sets after the split:")
print(f"X_train: {X_train.shape}, X_validate: {X_validate.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_validate: {y_validate.shape}, y_test: {y_test.shape}")



Now, one way to do things is to follow the steps:

1. Split the data into training, validation, and test sets (done)
2. For each parameter setting do the following:
    1. Train the model on the training set
    1. Validate the model on the validation set
3. Select the best parameters and train the model on the combined training and validation set
4. Train the model on the combined training and validation set and test it on the test set
5. Report the results using the values obtained in the test set
        


In [ ]:
best_score = 0

list_of_params = [.001, 0.01, 0.1, 1, 10, 100]

for gamma in list_of_params:
    for C in list_of_params:
        print(f"Testing gamma: {gamma}, C: {C} ... ", end="")
        svm = SVC(gamma=gamma, C=C).fit(X_train, y_train)
        score = svm.score(X_validate, y_validate)
        print(f"Score: {score}")
        if score > best_score:
            print(f"New best score: {score}")
            best_score = score
            best_params = {'C': C, 'gamma': gamma}
            
print(f'Over the validation set, the best params are: {best_params} with a score of: {best_score}')   

Now we rebuild the model on the combined training and validation set and test it with the test set: How does it behave on the test set?

In [ ]:
svm = SVC(**best_params).fit(X_train_validate, y_train_validate)
score = svm.score(X_test, y_test)

print(f'Over the test set the score is: {score}')

## Grid Search with Cross-Validation

For a better estimate of the generalization performance, instead of using a single split into a training and a validation set, we can use cross-validation to evaluate the performance of each parameter combination.

In [ ]:
digits = load_digits()

X_train, X_test, y_train, y_test = train_test_split(
    digits.data,
    digits.target, 
    random_state=100
)

As we will be using cross-validation, we don't need to split the data into training and validation sets. We can use the whole training set to train the model and evaluate it using cross-validation.

In [ ]:
best_score = 0

labels=[]
values=[]

list_of_params = [.001, 0.01, 0.1, 1, 10, 100]

for gamma in list_of_params:
    for C in list_of_params:
        print(f"Testing gamma: {gamma}, C: {C} ... ", end="")
        svm = SVC(gamma=gamma, C=C)
        score_array = cross_val_score(
            estimator=svm, 
            X=X_train, 
            y=y_train, 
            cv=10
        )
        print(f"Score: {score_array.mean()}")
        mean_score = score_array.mean()
        
        labels.append(f'C={C}/gamma={gamma}')
        values.append(score_array)
        
        if mean_score > best_score:
            print(f"New best score: {mean_score}")
            best_score = mean_score
            best_score_array = score_array
            best_params = {'C': C, 'gamma': gamma}

        
        
print(f'best param are {best_params} with a mean score {best_score} \n(the scores values where {best_score_array})')

Now we can use the best parameters to train the model and test it with the test set

In [ ]:
SVC(**best_params).fit(X_train, y_train).score(X_test, y_test)

As we used 6 values for each parameter, we have 36 different models. Furthermore, we used 10-fold cross-validation, so we have 360 different scores. The all set of values are

In [ ]:
values

Lets us plot these values and conclude that there are significant differences between the different parameter settings...

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))

ax.boxplot(values)
ax.set_xticklabels(labels=labels, rotation=75)
plt.ylabel("score")

plt.show()


## GridSearchCV
Because grid search with cross-validation is such a commonly used method to adjust parameters, scikit-learn provides the GridSearchCV class, which implements it in the form of an estimator. 

(https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [ ]:
list_of_params = [.001, 0.01, 0.1, 1, 10, 100]
param_grid = {
        'C': list_of_params, 
        'gamma': list_of_params
    } 

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data,
    digits.target, 
    random_state=0
)

clf = GridSearchCV(
    estimator=SVC(), 
    param_grid=param_grid, 
    cv=5, 
    return_train_score=True,
    verbose=3
).fit(X_train, y_train)

print(f'best score: { clf.best_score_}\n best params: {clf.best_params_}\n score over test: {clf.score(X_test, y_test)}\n\n Best estimator: {clf.best_estimator_}')

Now, we can use `pandas` to have a better look at the results

In [ ]:
results = pd.DataFrame(clf.cv_results_)

In [ ]:
results.sort_values(by='mean_test_score', ascending=False).head(10)

And now with data never seen

In [ ]:
clf = SVC(**clf.best_params_).fit(X_train, y_train)
clf.score(X_test, y_test)

## Search over spaces that are not grids

In [ ]:
param_grid = [
                {
                    'kernel': ['rbf'], 
                    'C': [1, 10, 100],
                    'gamma': [0.001, 0.01, 0.1]
                },
                {
                    'kernel': ['linear'],
                    'C': [0.001, 0.01, 0.1, 1, 10, 100]
                }
            ]

clf = GridSearchCV(
            estimator=SVC(), 
            param_grid=param_grid, 
            cv=5, 
            return_train_score=True).fit(X_train, y_train)

print(f'best score: { clf.best_score_}\n best params: {clf.best_params_}\n score over test: {clf.score(X_test, y_test)}\n\n Best estimator: {clf.best_estimator_}')

## Hyperparameter search without a full grid

A grid with *k*-fold cross-validation fits one model for every combination and every fold. The strategies below use fewer full trainings. On Iris the gain is the number of fits, not the clock time.

`C` is chosen by cross-validation on the training split from the start of the notebook. The test set is scored only after the search.

https://scikit-learn.org/stable/modules/grid_search.html


In [ ]:
print(f"train: {len(y_train)}   test: {len(y_test)} (locked)")


### Random search

Draw a fixed budget of values instead of every point on a grid. `n_iter` is the number of trainings (times the number of CV folds).

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html


In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    SVC(),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "gamma": loguniform(1e-3, 1e2),
    },
    n_iter=15,
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=3
).fit(X_train, y_train)

print(f"best parameters:        {random_search.best_params_}")
print(f"mean CV accuracy:       {random_search.best_score_:.3f}")
print(f"held-out test accuracy: {random_search.score(X_test, y_test):.3f}")


### Successive halving

In the grid and in random search, **every** candidate is trained on all of `X_train`. A bad pair `(C, gamma)` costs as much as a good one.

Successive halving changes the budget over time. Many candidates get a cheap look at a **small** slice of the training data. The worst half is dropped. The survivors get **twice as many rows**. That cut-and-double step repeats until only a few pairs remain, now trained on the full set.

With `factor=2` the shape of the search is:

| Round | How many candidates | How much data |
|-------|---------------------|---------------|
| 1     | many                | a small slice |
| 2     | half of those       | twice as many rows |
| 3     | half again          | twice as many rows |
| …     | a few               | all of `X_train` |

Most of the compute goes to pairs that already looked promising. Weak `C` and `gamma` leave after the cheap rounds. The locked test set is still unused until the search is over.

Arguments in the next cell:

- `SVC()` — the same model as above. The search draws `C` and `gamma`.
- `param_distributions={"C": loguniform(1e-3, 1e2), "gamma": loguniform(1e-3, 1e2)}` — both hyperparameters are drawn between \(10^{-3}\) and \(10^{2}\) on a log scale, the same range as the grid.
- `factor=2` — each round keeps the best half (`1/factor`) and doubles the resource. By default that resource is the number of training rows (`n_samples`).
- `cv=3` — 3-fold cross-validation on `X_train` only.
- `random_state=1` — same random draws if you re-run the cell.
- `n_jobs=-1` — use every CPU.
- `verbose=3` — print each fit as it runs.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html


In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

halving = HalvingRandomSearchCV(
    SVC(),
    param_distributions={
        "C": loguniform(1e-3, 1e2),
        "gamma": loguniform(1e-3, 1e2),
    },
    factor=2,
    cv=3,
    random_state=1,
    n_jobs=-1,
    verbose=3
)
halving.fit(X_train, y_train)
print(f"best parameters:        {halving.best_params_}")
print(f"mean CV accuracy:       {halving.best_score_:.3f}")
print(f"held-out test accuracy: {halving.score(X_test, y_test):.3f}")


### Bayesian optimization (Optuna)

Random search draws the next `(C, gamma)` **without looking** at the scores it already has. Trial 10 is as blind as trial 1.

Bayesian optimization does look. After each training it updates a cheap model of *which region of the `(C, gamma)` plane looked good*, then picks the next pair from there. Early trials still explore. Later trials spend more time near values that already scored well.

| Trial        | How the next pair is chosen |
|--------------|-----------------------------|
| first few    | almost at random (explore the plane) |
| later        | near pairs that already scored well (exploit) |

`n_trials` is the budget: how many SVMs you are willing to train (each one still uses 3-fold CV on `X_train`). Optuna's usual sampler is TPE (Tree-structured Parzen Estimator): it keeps two pictures of the search, *good trials* vs *the rest*, and draws the next point more like the good ones.

The locked test set is still unused until the study is over.

Parameters in the next cell:

- `suggest_float("C", 1e-3, 1e2, log=True)` and `suggest_float("gamma", 1e-3, 1e2, log=True)` — Optuna draws both SVM hyperparameters on a log scale, in the same range as the grid.
- `SVC(C=C, gamma=gamma)` — one model per trial, the same estimator as above.
- `cross_val_score(..., cv=3)` — 3-fold cross-validation on `X_train` only.
- `return scores.mean()` — the scalar Optuna tries to improve.
- `create_study(direction="maximize")` — higher mean accuracy is better. Use `"minimize"` for a loss.
- `TPESampler(seed=1)` — the sampler above. The seed makes the sequence of trials repeatable.
- `optimize(..., n_trials=15)` — stop after 15 complete trials.

https://optuna.readthedocs.io/


In [ ]:
#!pip install optuna
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 1e2, log=True)
    gamma = trial.suggest_float("gamma", 1e-3, 1e2, log=True)
    model = SVC(C=C, gamma=gamma)
    scores = cross_val_score(model, X_train, y_train, cv=3)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(
    objective, 
    n_trials=15, 
    show_progress_bar=True
)

final_model = SVC(**study.best_params).fit(X_train, y_train)
print(f"best parameters:        {study.best_params}")
print(f"mean CV accuracy:       {study.best_value:.3f}")
print(f"held-out test accuracy: {final_model.score(X_test, y_test):.3f}")
